### Set Up
```sh
conda create -n pymc python=3.14
conda activate pymc
python -m pip install -e .
pip install ipywidgets
pip install torch, pyro-ppl
```

In [1]:
import warnings

warnings.filterwarnings(
    "ignore",
    message='install "ipywidgets" for Jupyter support',
    category=UserWarning
)


In [2]:
from pmlearn.linear_model.base import LinearRegression
from pmlearn.linear_model.logistic import LogisticRegression
from pmlearn.factors.functional import FunctionalCPD
from pmlearn.linear_model.functional import FunctionalRegression

import pymc as pm
import pytensor.tensor as pt
import numpy as np


In [3]:
from sklearn.datasets import make_regression, make_classification

X, y, true_coef = make_regression(
    n_samples=100,
    n_features=3,
    n_informative=3,
    noise=10.0,
    coef=True,
    random_state=42
)

print("True coef:", true_coef)


True coef: [28.20345726 75.06147516 17.74395438]


# LinearRegression

In [4]:
model = LinearRegression()
model.fit(X, y)


Output()

Convergence achieved at 105500
Interrupted at 105,499 [52%]: Average Loss = 15,634


LinearRegression()

In [5]:
result = model.predict(X[:5])
result


Sampling: [y]


Output()

array([[-22.77558071,  -3.09072558,  44.39125339, -52.41621052,
        -14.4105199 ],
       [ -2.36001696,  -5.00730582,  60.500942  ,  -4.50040731,
          3.75870066],
       [-13.13757431, -13.05982838,  57.03340646, -48.81636039,
          8.53991068],
       ...,
       [-45.86304223, -14.42528789,  52.58072286,   2.39034889,
        -11.43575931],
       [  8.24723097,  16.76476523,  53.89226362, -54.4756918 ,
         -2.42728977],
       [-11.24351016,   3.41033853,  31.36199115, -16.93280871,
        -42.75148903]], shape=(10000, 5))

# LogisticRegression

In [6]:
X, y = make_classification(
    n_samples=100,
    n_features=20,
    random_state=42
)

cats = np.zeros(X.shape[0], dtype=np.int64)


In [7]:
model = LogisticRegression()


In [8]:
model.fit(X, y, cats)


Output()

Finished [100%]: Average Loss = 66.839


LogisticRegression()

# FunctioanlCPD

In [9]:
import pandas as pd
import pymc as pm
from pmlearn.factors import FunctionalCPD
cpd = FunctionalCPD(
    variable='x3',
    fn=lambda parent_sample: pm.Normal.dist(
        mu=1.0 + 0.2 * parent_sample['x1'] + 0.3 * parent_sample['x2'],
        sigma=1,
    ),
    parents=['x1', 'x2'],
)
parent_samples = pd.DataFrame({'x1': [5, 10], 'x2': [1, -1]})
cpd.sample(2, parent_samples)


array([2.65811329, 2.66336192])

# FunctionalRegression

In [10]:
def make_linear_formula(n_features):
    def formula(X):
        # scalar
        intercept = pm.Normal(
            "intercept",
            mu=0.0,
            sigma=10.0,
        )

        # shape: (n_features,)
        beta = pm.Normal(
            "beta",
            mu=0.0,
            sigma=5.0,
            shape=(n_features,),
        )

        # X:    (n_samples, n_features)
        # beta: (n_features,)
        # mu:   (n_samples,)
        mu = intercept + pt.dot(X, beta)

        return mu

    return formula



X = np.asarray(X, dtype=float)
y = np.asarray(y, dtype=float).reshape(-1)

formula = make_linear_formula(X.shape[1])


In [11]:
model = FunctionalRegression(
    formula=formula,
    sigma=1.0,
)


In [12]:
model.fit(X, y)


Output()

Finished [100%]: Average Loss = 99.226


,formula,<function mak...0023C2846DBC0>
,sigma,1.0
Name,Type,Value
n_features_in_,int,20


In [13]:
result = model.predict_proba(X[:5])
result


Sampling: [y]


Output()

array([0.37989403, 0.05049606, 0.79083819, 0.94638068, 0.19845893])

In [14]:
import torch
import pyro
import pyro.distributions as dist
from pmlearn.linear_model.functionalpyro import FunctionalRegression as FunctionalRegressionPyro

def make_linear_formula(n_features: int):
    if n_features <= 0:
        raise ValueError("`n_features` must be positive.")

    def formula(X: torch.Tensor) -> torch.Tensor:
        # Scalar global latent variable.
        intercept = pyro.sample(
            "intercept",
            dist.Normal(
                X.new_tensor(0.0),
                X.new_tensor(10.0),
            ),
        )

        # One n_features-dimensional global latent variable.
        beta = pyro.sample(
            "beta",
            dist.Normal(
                X.new_zeros(n_features),
                X.new_full((n_features,), 5.0),
            ).to_event(1),
        )

        # X:    (n_samples, n_features)
        # beta: (n_features,)
        # mu:   (n_samples,)
        mu = intercept + X @ beta

        return mu

    return formula


In [15]:
X = np.asarray(X, dtype=np.float32)
y = np.asarray(y, dtype=np.float32).reshape(-1)

formula = make_linear_formula(X.shape[1])

model = FunctionalRegressionPyro(
    formula=formula,
    sigma=1.0,
    dtype=torch.float32,
    random_state=42,
)

model.fit(
    X,
    y,
    inference_type="advi",
    minibatch_size=None,
    inference_args={
        "num_steps": 5000,
        "lr": 0.01,
        "num_particles": 1,
    },
)

prediction_mean, prediction_std = model.predict(
    X,
    return_std=True,
    num_samples=2000,
)

print(prediction_mean.shape)
print(prediction_std.shape)


(100,)
(100,)
